# Phase 1 -- Notebook p1_03: Model A trainer (resumable)

**What this notebook does.** Trains Model A (163M params) on the real legal
corpus, with checkpoint/resume, KV-cache generation, and a submission
packer -- the actual `legallm` trainer, not a bench harness.

**Two modes, one code path** (see Section 2):
- `SMOKE_MODE = True` (today, Sat): a short, cheap run -- a few hundred
  steps or ~45 minutes, whichever comes first -- to prove the *whole*
  pipeline works: data loads, loss goes down, a checkpoint saves and can
  be loaded back, and generation produces text. Checked locally with
  `evaluate_phase1.py` -- not submitted to Codabench (submissions are
  capped at 6/phase, too scarce to spend on a deliberately undertrained
  model).
- `SMOKE_MODE = False` (tomorrow onward): the real run, same code, a much
  larger time/token budget, and it resumes from wherever the smoke run
  (or the previous session) left off.

**What changed since the bench notebook:** gradient checkpointing is now
on for training (Section 4). The bench notebook found Model A's designed
`max_seq_len=2048` OOMs at any batch size on a single T4 without it --
see the plan doc for the numbers. This notebook re-checks the actual
fitting batch size at the real training sequence length *with*
checkpointing on, empirically, rather than assuming the bench notebook's
un-checkpointed numbers still apply (Section 5).

**Runs on:** Kaggle GPU session (T4 x2, using GPU 0 only -- see the note
in Section 1 on why this notebook doesn't attempt multi-GPU DDP today).

**Attach as input data:** `legallm-phase1-data`, `legallm-tokenizers`,
and (only once one exists) `legallm-checkpoints` for resume.


## 1. Imports & GPU check

In [1]:
import itertools
import json
import math
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  "
          f"({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")

# Kaggle's T4x2 sessions give two GPUs, but real multi-GPU training needs
# DDP, which needs a multi-process launcher (torch.multiprocessing.spawn
# or torchrun) -- awkward and fragile inside a single Jupyter kernel.
# Phase 1's budget (3-12 GPU-h per session, per the plan doc) is workable
# on one GPU, so this notebook deliberately trains on GPU 0 only and
# keeps that complexity out of the critical path. Revisit DDP only if a
# later phase's token budget makes single-GPU throughput the bottleneck.
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    print("\nWARNING: no GPU detected. Switch this notebook's accelerator "
          "to GPU T4 x2 in Settings before running the rest of this notebook.")

INPUT_DATA_DIR = Path("/kaggle/input/datasets/jatiniitk/legallm-phase1-data")
INPUT_TOKENIZER_DIR = Path("/kaggle/input/datasets/jatiniitk/legallm-tokenizers")
INPUT_CHECKPOINT_DIR = Path("/kaggle/input/datasets/jatiniitk/legallm-checkpoints")  # may not exist yet -- checked before use
OUTPUT_DIR = Path("/kaggle/working/phase1_model_a")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

print("\nlegallm-phase1-data contents:", sorted(p.name for p in INPUT_DATA_DIR.glob("*")))
print("legallm-tokenizers contents:", sorted(p.name for p in INPUT_TOKENIZER_DIR.glob("*")))


PyTorch version: 2.10.0+cu128
CUDA available: True
  GPU 0: Tesla T4  (15.6 GB)
  GPU 1: Tesla T4  (15.6 GB)

legallm-phase1-data contents: ['clean_acts.txt', 'clean_high_courts.txt', 'clean_supreme_court.txt', 'clean_tribunals.txt', 'dev_prompts.json', 'dev_truth.json']
legallm-tokenizers contents: ['t1_bpe.json', 't2_train.log', 't2_unigram.model', 't2_unigram.vocab']


## 2. Run mode & config

Flip `SMOKE_MODE` to `False` tomorrow for the real run -- everything else
in this notebook adapts automatically (time budget, checkpoint interval,
target step count). `arch_id = "A"` matches the plan doc's checkpoint
file-naming convention: `{arch_id}_step{step}_tokens{tokens_seen}.pt`.


In [2]:
SMOKE_MODE = False  # <-- the one flag to change for tomorrow's real run

MODEL_A_CONFIG = dict(
    vocab_size=32_000,    # matches T1 (BPE), see p1_01_tokenizer.ipynb
    d_model=896,
    n_layer=16,
    n_head=14,
    n_kv_head=2,           # 7:1 GQA ratio
    d_head=64,
    d_ff=2432,
    max_seq_len=2048,      # the architecture's designed capacity (RoPE cache size)
)
ARCH_ID = "A"

# Training sequence length is deliberately shorter than max_seq_len. The
# bench notebook found 2048 OOMs outright on a T4; 1024 is the target we
# picked to keep most legal documents from being truncated to nothing
# while still fitting in memory with gradient checkpointing on (verified
# empirically in Section 5, not just assumed here).
TRAIN_SEQ_LEN = 1024

# Optimization hyperparameters. Peak LR and warmup follow common practice
# for this model scale (roughly GPT-2-small/medium territory); these are
# starting points, not tuned -- the fp16 stability check in the bench
# notebook only ruled out structural instability, not that this exact LR
# is optimal on real data.
PEAK_LR = 3e-4
MIN_LR = 3e-5
WARMUP_STEPS = 200
WEIGHT_DECAY = 0.1
GRAD_CLIP_NORM = 1.0

if SMOKE_MODE:
    TIME_BUDGET_MIN = 45          # stop well inside Kaggle's 12h session cap
    MAX_STEPS = 400                # whichever of time/steps comes first
    CHECKPOINT_EVERY_STEPS = 100
    GRAD_ACCUM_STEPS = 1            # kept small so the smoke run is fast to iterate on
    LR_DECAY_STEPS = 400            # decay fully within the smoke run's own short life -- just proves the schedule works
else:
    # Deliberately short of the 12h session cap: with checkpoints now
    # pushing periodically (Section 8), training in several shorter,
    # resumable chunks banks the same total progress as one long session
    # -- without risking losing all of it to one crash, the way an
    # earlier uncheckpointed 11h run did. Raise this per-session if your
    # remaining weekly GPU quota and schedule comfortably allow it.
    TIME_BUDGET_MIN = 300           # 5h -- leaves headroom in a tight weekly quota for other models (B, C, D)
    MAX_STEPS = 10 ** 9             # effectively unbounded -- the time budget is what actually stops each session
    CHECKPOINT_EVERY_STEPS = 500    # roughly every ~30 min at expected throughput, per the plan doc's contract note
    GRAD_ACCUM_STEPS = 4            # larger effective batch for the real run
    # The cosine schedule below decays over a FIXED step horizon, not over
    # MAX_STEPS -- MAX_STEPS is deliberately huge (real training spans
    # multiple resumed sessions across several days, per the plan doc's
    # day-by-day table, so there's no single session-local "total steps").
    # 8,000 steps is a placeholder horizon sized for roughly the token
    # budget the plan doc targets for Model A's main run (~1.5B tokens);
    # revisit this number once Section 5's real fitting batch size is
    # known and tokens-per-step can be computed exactly.
    LR_DECAY_STEPS = 8_000

print(f"SMOKE_MODE = {SMOKE_MODE}")
print(f"Model A config: {json.dumps(MODEL_A_CONFIG, indent=2)}")
print(f"TRAIN_SEQ_LEN={TRAIN_SEQ_LEN}  TIME_BUDGET_MIN={TIME_BUDGET_MIN}  "
      f"MAX_STEPS={MAX_STEPS}  GRAD_ACCUM_STEPS={GRAD_ACCUM_STEPS}  LR_DECAY_STEPS={LR_DECAY_STEPS}")


SMOKE_MODE = False
Model A config: {
  "vocab_size": 32000,
  "d_model": 896,
  "n_layer": 16,
  "n_head": 14,
  "n_kv_head": 2,
  "d_head": 64,
  "d_ff": 2432,
  "max_seq_len": 2048
}
TRAIN_SEQ_LEN=1024  TIME_BUDGET_MIN=300  MAX_STEPS=1000000000  GRAD_ACCUM_STEPS=4  LR_DECAY_STEPS=8000


## 3. Tokenizer

Loads T1 (byte-level BPE), the tokenizer this run's `vocab_size=32,000`
and special tokens were built around (see `p1_01_tokenizer.ipynb`).


In [3]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(str(INPUT_TOKENIZER_DIR / "t1_bpe.json"))
EOS_ID = tokenizer.token_to_id("<eos>")
assert EOS_ID is not None, "T1 tokenizer is missing the <eos> special token -- check p1_01's ALL_SPECIAL_TOKENS."

print(f"Tokenizer vocab size: {tokenizer.get_vocab_size()}")
print(f"<eos> id: {EOS_ID}")
print("Round trip check:", tokenizer.decode(tokenizer.encode("Section 302 of the Indian Penal Code.").ids))


Tokenizer vocab size: 32000
<eos> id: 2
Round trip check: Section 302 of the Indian Penal Code.


## 4. The model -- with gradient checkpointing

Same blocks as `p1_02_bench.ipynb` (kept inline here rather than an
importable `legallm` package for now -- see the note at the end of this
notebook on packaging that as `legallm-src` once the interfaces stop
moving). Two additions match the plan doc's fixed function-name contract:
`model.encode(...)` for a raw hidden-states call, and `model.generate(...)`
for KV-cache decoding.

**Gradient checkpointing**, added per the bench notebook's finding:
instead of keeping every layer's activations in memory for the backward
pass, each `Block`'s forward is wrapped in `torch.utils.checkpoint`,
which discards activations after the forward pass and recomputes them
during backward. This trades roughly 20-30% more compute for a large cut
in activation memory -- exactly the trade we want, since Section 5 showed
this GPU is memory-bound at the sequence length we need, not compute-bound.
It's only applied during training (`model.training`); at inference/generation
time there's no backward pass, so recomputing would be pure waste.


In [4]:
class RMSNorm(nn.Module):
    """RMSNorm instead of LayerNorm: one learned scale, no learned bias,
    no mean-subtraction -- cheaper, and the modern-architecture standard
    this whole design follows (Llama, Mistral, etc.).
    """
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        norm = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight.float()).to(dtype)


def build_rope_cache(seq_len, d_head, theta=10000.0, device="cpu"):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    rotated = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return rotated.flatten(-2)


try:
    _probe = torch.randn(1, 1, 2, 4)
    F.scaled_dot_product_attention(_probe, _probe, _probe, enable_gqa=True)
    _SDPA_HAS_ENABLE_GQA = True
except TypeError:
    _SDPA_HAS_ENABLE_GQA = False
print(f"SDPA native enable_gqa available: {_SDPA_HAS_ENABLE_GQA}")


class GQAAttention(nn.Module):
    """Grouped-query attention: fewer KV heads than query heads, which
    shrinks the KV cache proportionally (here, 7x) at a small quality cost.
    """
    def __init__(self, d_model, n_head, n_kv_head, d_head):
        super().__init__()
        self.n_head, self.n_kv_head, self.d_head = n_head, n_kv_head, d_head
        self.q_proj = nn.Linear(d_model, n_head * d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.o_proj = nn.Linear(n_head * d_head, d_model, bias=False)

    def forward(self, x, cos, sin, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if cache is not None:
            k, v = cache.update(k, v)

        is_causal = q.shape[2] > 1

        if _SDPA_HAS_ENABLE_GQA:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal, enable_gqa=True)
        else:
            n_rep = self.n_head // self.n_kv_head
            k_rep = k.repeat_interleave(n_rep, dim=1)
            v_rep = v.repeat_interleave(n_rep, dim=1)
            out = F.scaled_dot_product_attention(q, k_rep, v_rep, is_causal=is_causal)

        out = out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head)
        return self.o_proj(out)


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class Block(nn.Module):
    """One transformer layer: pre-norm attention, pre-norm MLP, both as
    residual additions.
    """
    def __init__(self, d_model, n_head, n_kv_head, d_head, d_ff):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQAAttention(d_model, n_head, n_kv_head, d_head)
        self.mlp_norm = RMSNorm(d_model)
        self.mlp = SwiGLU(d_model, d_ff)

    def _forward_inner(self, x, cos, sin, cache):
        x = x + self.attn(self.attn_norm(x), cos, sin, cache)
        x = x + self.mlp(self.mlp_norm(x))
        return x

    def forward(self, x, cos, sin, cache=None, use_checkpoint=False):
        # Checkpointing needs a cache-free forward (no in-place cache
        # mutation across a recomputed backward pass) and only makes
        # sense during training, never during KV-cache generation.
        if use_checkpoint and cache is None and self.training:
            return grad_checkpoint(self._forward_inner, x, cos, sin, None, use_reentrant=False)
        return self._forward_inner(x, cos, sin, cache)


class KVCache:
    """A pre-allocated, fixed-size cache for one layer's keys and values."""
    def __init__(self, batch_size, n_kv_head, max_seq_len, d_head, device, dtype):
        shape = (batch_size, n_kv_head, max_seq_len, d_head)
        self.k = torch.zeros(shape, device=device, dtype=dtype)
        self.v = torch.zeros(shape, device=device, dtype=dtype)
        self.pos = 0

    def update(self, k, v):
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


class MiniGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([
            Block(config["d_model"], config["n_head"], config["n_kv_head"],
                  config["d_head"], config["d_ff"])
            for _ in range(config["n_layer"])
        ])
        self.norm_f = RMSNorm(config["d_model"])
        self.head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        self.head.weight = self.embed.weight  # weight tying

        cos, sin = build_rope_cache(config["max_seq_len"], config["d_head"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config["n_layer"]))

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def new_kv_caches(self, batch_size, device, dtype):
        return [
            KVCache(batch_size, self.config["n_kv_head"], self.config["max_seq_len"],
                    self.config["d_head"], device, dtype)
            for _ in range(self.config["n_layer"])
        ]

    def encode(self, input_ids, attention_mask=None, mode="causal"):
        """Contract method: raw hidden states, no LM head. attention_mask
        is accepted for interface compatibility with later phases (e.g. a
        padded batch) but unused here -- Phase 1 pretraining always packs
        full, unpadded windows, so every position is valid.
        """
        assert mode == "causal", "MiniGPT is decoder-only; only causal encoding is supported."
        B, T = input_ids.shape
        x = self.embed(input_ids)
        cos = self.rope_cos[:T].to(x.device)
        sin = self.rope_sin[:T].to(x.device)
        use_ckpt = self.training and USE_GRAD_CHECKPOINTING
        for blk in self.blocks:
            x = blk(x, cos, sin, cache=None, use_checkpoint=use_ckpt)
        return self.norm_f(x)

    def forward(self, ids, caches=None):
        if caches is None:
            return self.head(self.encode(ids))
        # Generation path (KV cache present): identical to encode() but
        # threading a per-layer cache through and offsetting RoPE by how
        # much of the cache is already filled.
        B, T = ids.shape
        x = self.embed(ids)
        pos_offset = caches[0].pos
        cos = self.rope_cos[pos_offset:pos_offset + T].to(x.device)
        sin = self.rope_sin[pos_offset:pos_offset + T].to(x.device)
        for i, blk in enumerate(self.blocks):
            x = blk(x, cos, sin, cache=caches[i], use_checkpoint=False)
        x = self.norm_f(x)
        return self.head(x)

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy", top_p=0.9, temperature=1.0):
        """Contract method. `sampler`: "greedy" (argmax, deterministic --
        what the smoke-test submission uses) or "top_p" (nucleus sampling).
        Stops early (per sequence, via a done mask) at <eos> or max_seq_len.
        """
        assert kv_cache, "This notebook's generate() only implements the KV-cache path -- it's what training will actually ship."
        self.eval()
        device, dtype = input_ids.device, next(self.parameters()).dtype

        # A prompt longer than the model's context has nowhere to put
        # max_new_tokens (or, if it's longer than max_seq_len outright,
        # RoPE's precomputed cos/sin tables don't even cover it -- a hard
        # crash, not a quality issue). Truncate to the most recent tokens
        # -- the ones closest to where generation continues -- rather
        # than letting this crash the whole generation loop over one long
        # dev-set example.
        max_prompt_len = self.config["max_seq_len"] - max_new_tokens
        if input_ids.shape[1] > max_prompt_len:
            print(f"  [generate() warning: prompt has {input_ids.shape[1]} tokens, "
                  f"truncating to the last {max_prompt_len} to leave room for "
                  f"{max_new_tokens} new tokens within max_seq_len={self.config['max_seq_len']}]")
            input_ids = input_ids[:, -max_prompt_len:]

        B, T0 = input_ids.shape
        max_total = min(T0 + max_new_tokens, self.config["max_seq_len"])

        caches = self.new_kv_caches(batch_size=B, device=device, dtype=dtype)
        generated = input_ids
        done = torch.zeros(B, dtype=torch.bool, device=device)

        logits = self.forward(input_ids, caches=caches)  # prefill: process the whole prompt at once
        next_ids = self._sample(logits[:, -1, :], sampler, top_p, temperature)

        while generated.shape[1] < max_total:
            next_ids = torch.where(done, torch.full_like(next_ids, EOS_ID), next_ids)
            generated = torch.cat([generated, next_ids.unsqueeze(1)], dim=1)
            done = done | (next_ids == EOS_ID)
            if done.all():
                break
            logits = self.forward(next_ids.unsqueeze(1), caches=caches)  # one new token at a time
            next_ids = self._sample(logits[:, -1, :], sampler, top_p, temperature)

        self.train()
        return generated

    @staticmethod
    def _sample(logits, sampler, top_p, temperature):
        if sampler == "greedy":
            return logits.argmax(dim=-1)
        probs = F.softmax(logits / temperature, dim=-1)
        sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
        cumulative = sorted_probs.cumsum(dim=-1)
        cutoff = (cumulative <= top_p).sum(dim=-1, keepdim=True).clamp(min=1)
        mask = torch.arange(sorted_probs.shape[-1], device=logits.device)[None, :] < cutoff
        sorted_probs = sorted_probs * mask
        sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
        picked = torch.multinomial(sorted_probs, num_samples=1).squeeze(-1)
        return sorted_idx.gather(-1, picked.unsqueeze(-1)).squeeze(-1)


def build_model(config):
    """Contract function: the one place that turns a config dict into a
    model instance, so every later phase constructs Model A the same way.
    """
    return MiniGPT(config)


USE_GRAD_CHECKPOINTING = True  # the Section-4 decision; flip off only to debug, never for a real run
model = build_model(MODEL_A_CONFIG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model A parameter count (tied embeddings): {n_params / 1e6:.1f}M")
assert 50e6 <= n_params <= 300e6, "outside the competition's allowed parameter range!"


SDPA native enable_gqa available: True
Model A parameter count (tied embeddings): 162.7M


## 5. Re-check the fitting batch size, with checkpointing on

The bench notebook's throughput sweep did **not** have gradient
checkpointing enabled, so its numbers don't directly answer "what fits at
`TRAIN_SEQ_LEN=1024` now". A short, targeted re-check here -- rather than
assuming -- is cheap (a few seconds) and avoids discovering an OOM 20
minutes into a real training run.


In [5]:
def try_batch_size(batch_size, seq_len, n_iters=3):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    try:
        batch = torch.randint(0, MODEL_A_CONFIG["vocab_size"], (batch_size, seq_len), device=DEVICE)
        targets = torch.randint(0, MODEL_A_CONFIG["vocab_size"], (batch_size, seq_len), device=DEVICE)
        probe_optim = torch.optim.AdamW(model.parameters(), lr=1e-4)  # same optimizer-state memory shape as real training
        probe_scaler = torch.cuda.amp.GradScaler()
        for _ in range(n_iters):
            probe_optim.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(batch)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            probe_scaler.scale(loss).backward()
            probe_scaler.step(probe_optim)
            probe_scaler.update()
        torch.cuda.synchronize()
        peak_gb = torch.cuda.max_memory_allocated() / 1e9
        del probe_optim  # free its state before the next candidate batch size
        return peak_gb
    except torch.cuda.OutOfMemoryError:
        return None


CANDIDATE_BATCH_SIZES = [4, 8, 16, 24, 32]
MEMORY_BUDGET_GB = 13.0
fitting_batch_size = None
for bs in CANDIDATE_BATCH_SIZES:
    peak_gb = try_batch_size(bs, TRAIN_SEQ_LEN)
    status = "OOM" if peak_gb is None else (f"{peak_gb:.2f} GB" + (" [fits]" if peak_gb <= MEMORY_BUDGET_GB else " [over budget]"))
    print(f"  batch={bs:3d}  seq_len={TRAIN_SEQ_LEN}  (checkpointing on) -> {status}")
    if peak_gb is not None and peak_gb <= MEMORY_BUDGET_GB:
        fitting_batch_size = bs

assert fitting_batch_size is not None, (
    "Nothing fit under the memory budget even at the smallest candidate batch size -- "
    "lower TRAIN_SEQ_LEN or MEMORY_BUDGET_GB's assumptions and re-run this cell."
)
MICRO_BATCH_SIZE = fitting_batch_size
EFFECTIVE_BATCH_SIZE = MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS
print(f"\nUsing micro-batch size {MICRO_BATCH_SIZE} x {GRAD_ACCUM_STEPS} grad-accum steps "
      f"= effective batch size {EFFECTIVE_BATCH_SIZE} at seq_len={TRAIN_SEQ_LEN}.")


/tmp/ipykernel_23/1644024308.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  probe_scaler = torch.cuda.amp.GradScaler()


  batch=  4  seq_len=1024  (checkpointing on) -> 3.95 GB [fits]
  batch=  8  seq_len=1024  (checkpointing on) -> 5.72 GB [fits]
  batch= 16  seq_len=1024  (checkpointing on) -> 9.41 GB [fits]
  batch= 24  seq_len=1024  (checkpointing on) -> 13.10 GB [over budget]
  batch= 32  seq_len=1024  (checkpointing on) -> OOM

Using micro-batch size 16 x 4 grad-accum steps = effective batch size 64 at seq_len=1024.


## 6. Data pipeline

Tokenizes the cleaned legal-text shards on the fly rather than
pre-encoding the whole corpus into memory up front -- the fast (Rust)
tokenizer keeps this cheap relative to the GPU forward/backward pass, and
it means training can start immediately without a separate
multi-GB-file encoding step. Documents are separated by `<eos>` in the
token stream so the model learns document boundaries; text is drawn from
all four sources in a fixed round-robin, which is a simple, defensible
starting mix -- revisit if the Sun Sep 27 data-mix ablations suggest a
better ratio.


In [6]:
class TokenStreamDataset(torch.utils.data.IterableDataset):
    """Yields (input_ids, target_ids) pairs of length TRAIN_SEQ_LEN, built
    from a rolling buffer of tokenized text pulled from `file_paths` in
    round-robin order. Loops forever over the files -- the training loop
    (Section 7), not this dataset, decides when to stop.
    """
    def __init__(self, file_paths, tokenizer, seq_len):
        self.file_paths = file_paths
        self.tokenizer = tokenizer
        self.seq_len = seq_len

    def __iter__(self):
        buffer = []
        file_iters = [self._read_lines(p) for p in self.file_paths]
        for line_iter in itertools.cycle(file_iters):
            line = next(line_iter, None)
            if line is None:
                continue  # this source is exhausted for now; keep round-robining the others
            ids = self.tokenizer.encode(line).ids
            if not ids:
                continue
            buffer.extend(ids)
            buffer.append(EOS_ID)
            while len(buffer) >= self.seq_len + 1:
                window = buffer[:self.seq_len + 1]
                buffer = buffer[self.seq_len + 1:]
                x = torch.tensor(window[:-1], dtype=torch.long)
                y = torch.tensor(window[1:], dtype=torch.long)
                yield x, y

    @staticmethod
    def _read_lines(path):
        # A generator, not a list -- so re-reading a file from the start
        # (once itertools.cycle wraps around) is just calling this again,
        # never holding the whole file in memory.
        while True:
            with open(path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        yield line
            # File exhausted: loop back and reopen it (next epoch over this source).


TRAIN_FILES = sorted(INPUT_DATA_DIR.glob("clean_*.txt"))
assert TRAIN_FILES, f"No clean_*.txt files found under {INPUT_DATA_DIR} -- check the attached dataset."
print("Training on:", [p.name for p in TRAIN_FILES])

train_dataset = TokenStreamDataset(TRAIN_FILES, tokenizer, TRAIN_SEQ_LEN)
# num_workers=0: tokenizing with the Rust-backed `tokenizers` library is
# fast enough relative to a 163M-param forward+backward that a separate
# worker process isn't worth the added complexity (and the round-robin
# file cycling above isn't worker-safe as written -- each worker would
# duplicate the whole stream). Revisit only if profiling shows the data
# loader, not the GPU, is the bottleneck.
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=MICRO_BATCH_SIZE)

# Quick sanity check: pull one batch and confirm shapes/dtypes before
# committing to a training loop built on top of it.
_sample_x, _sample_y = next(iter(train_loader))
print(f"Sample batch: x={tuple(_sample_x.shape)} {_sample_x.dtype}, y={tuple(_sample_y.shape)} {_sample_y.dtype}")


Training on: ['clean_acts.txt', 'clean_high_courts.txt', 'clean_supreme_court.txt', 'clean_tribunals.txt']
Sample batch: x=(16, 1024) torch.int64, y=(16, 1024) torch.int64


## 7. Checkpoint save/load (contract functions)

Exactly the dict shape and function signature from the plan doc's
checkpoint contract, so Phases 2-7 -- or a fresh chat session with no
memory of this one -- can load a Phase 1 checkpoint without guessing:
`save_checkpoint(path, model, optim, sched, scaler, step, tokens_seen, config)`,
dict = `{model_state, optim_state, scheduler_state, scaler_state, step,
tokens_seen, rng_state, config, arch_id}`. File naming:
`{arch_id}_step{step}_tokens{tokens_seen}.pt`.


In [7]:
def save_checkpoint(path, model, optim, sched, scaler, step, tokens_seen, config):
    torch.save({
        "model_state": model.state_dict(),
        "optim_state": optim.state_dict(),
        "scheduler_state": sched.state_dict() if sched is not None else None,
        "scaler_state": scaler.state_dict(),
        "step": step,
        "tokens_seen": tokens_seen,
        "rng_state": torch.get_rng_state(),
        "config": config,
        "arch_id": ARCH_ID,
    }, path)


def load_checkpoint(path, map_location):
    return torch.load(path, map_location=map_location)


def checkpoint_path_for(step, tokens_seen):
    return CHECKPOINT_DIR / f"{ARCH_ID}_step{step}_tokens{tokens_seen}.pt"


print("save_checkpoint / load_checkpoint / checkpoint_path_for defined.")


save_checkpoint / load_checkpoint / checkpoint_path_for defined.


## 8. Kaggle Dataset push helper -- used *during* training, not just at the end

**Why this section exists:** an earlier run of this notebook died mid-session
(interactive "Run All" got interrupted) after training for several hours,
and lost all of it -- because checkpoints only got pushed to
`legallm-checkpoints` in the very last cell, which the run never reached.
Local `/kaggle/working` files do not survive a killed session either, so
nothing was recoverable.

**Fix:** `push_checkpoint_dataset()` is defined here so Section 10's
training loop can call it every `CHECKPOINT_EVERY_STEPS`, not only once
at the end. A session dying mid-run now loses at most one checkpoint
interval, not the whole run. `prune_old_checkpoints()` keeps only the
most recent couple of local checkpoints before each push -- without it,
every push would re-upload every checkpoint ever saved this run (each is
model + optimizer state, a few hundred MB to a couple GB), growing
without bound over a long run.


In [8]:
import shutil
import subprocess

from kaggle_secrets import UserSecretsClient

KAGGLE_USERNAME = "jatiniitk"
CHECKPOINT_DATASET_SLUG = "legallm-checkpoints"


def get_kaggle_token():
    """Reads the KAGGLE_API_TOKEN secret and writes it where the `kaggle`
    CLI expects it. Returns None (and explains why) if the secret isn't
    available -- callers treat that as "skip the push", not a crash.
    """
    try:
        token = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
    except Exception as exc:
        print(f"Could not read the 'KAGGLE_API_TOKEN' secret ({exc}). "
              f"Add it under Add-ons -> Secrets (and check the box on!) to enable checkpoint pushes.")
        return None
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    token_path = kaggle_dir / "access_token"
    token_path.write_text(token)
    token_path.chmod(0o600)
    return token


def prune_old_checkpoints(keep=2):
    """Deletes local checkpoint files older than the most recent `keep`,
    so CHECKPOINT_DIR (and every push's upload size) stays bounded over a
    long run instead of growing by one multi-hundred-MB file every
    CHECKPOINT_EVERY_STEPS.
    """
    ckpts = sorted(CHECKPOINT_DIR.glob(f"{ARCH_ID}_step*.pt"),
                   key=lambda p: int(p.stem.split("_step")[1].split("_tokens")[0]))
    for p in ckpts[:-keep] if len(ckpts) > keep else []:
        p.unlink()


def push_checkpoint_dataset(message):
    """Pushes whatever is currently in OUTPUT_DIR (checkpoints, and
    runs.csv/predictions once Sections 11-14 have written them) as a new
    version of the legallm-checkpoints Kaggle Dataset. Safe to call
    repeatedly and cheap relative to a training step -- each call is a
    small, incremental version push, not a full re-upload of the
    training data.
    """
    if get_kaggle_token() is None:
        print("Skipping checkpoint push: no KAGGLE_API_TOKEN secret.")
        return False

    dataset_id = f"{KAGGLE_USERNAME}/{CHECKPOINT_DATASET_SLUG}"
    metadata = {"title": CHECKPOINT_DATASET_SLUG, "id": dataset_id, "licenses": [{"name": "CC0-1.0"}]}
    with open(OUTPUT_DIR / "dataset-metadata.json", "w") as f:
        json.dump(metadata, f)

    exists = subprocess.run(
        ["kaggle", "datasets", "status", dataset_id], capture_output=True, text=True,
    ).returncode == 0

    if exists:
        cmd = ["kaggle", "datasets", "version", "-p", str(OUTPUT_DIR), "-m", message, "-r", "zip"]
    else:
        print(f"Dataset {dataset_id} does not exist yet -> creating it.")
        cmd = ["kaggle", "datasets", "create", "-p", str(OUTPUT_DIR), "-r", "zip"]

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Checkpoint push failed: {result.stderr.strip()[-500:]}")
        return False
    print(f"Pushed checkpoint dataset ({message}).")
    return True


print("push_checkpoint_dataset() / prune_old_checkpoints() defined -- "
      "called periodically during training (Section 10), not just at the end.")


push_checkpoint_dataset() / prune_old_checkpoints() defined -- called periodically during training (Section 10), not just at the end.


## 9. Optimizer, schedule, and resume

Looks for an existing Model A checkpoint in the attached
`legallm-checkpoints` dataset (won't exist yet on the very first run --
that's expected and handled) and resumes from the latest step if found,
rather than always starting from scratch.


In [9]:
optimizer = torch.optim.AdamW(model.parameters(), lr=PEAK_LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
scaler = torch.cuda.amp.GradScaler()


def get_lr(step):
    """Linear warmup for WARMUP_STEPS, then cosine decay from PEAK_LR down
    to MIN_LR over LR_DECAY_STEPS, then held flat at MIN_LR. A fixed
    function of step (not a torch LR scheduler object) so it's trivial to
    resume exactly -- no scheduler state to restore, just recompute from
    the resumed step.

    Deliberately decays over LR_DECAY_STEPS, not MAX_STEPS: MAX_STEPS is
    the loop's stop condition (huge/effectively-unbounded in main mode,
    since the real run spans multiple resumed sessions), while
    LR_DECAY_STEPS is a fixed schedule horizon independent of how many
    sessions it takes to get there. Using MAX_STEPS here directly would
    leave the LR pinned near PEAK_LR for the entire main run.
    """
    if step < WARMUP_STEPS:
        return PEAK_LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, LR_DECAY_STEPS - WARMUP_STEPS)
    progress = min(progress, 1.0)
    cosine = 0.5 * (1 + math.cos(math.pi * progress))
    return MIN_LR + (PEAK_LR - MIN_LR) * cosine


start_step, start_tokens_seen = 0, 0
if INPUT_CHECKPOINT_DIR.exists():
    existing = sorted(INPUT_CHECKPOINT_DIR.glob(f"{ARCH_ID}_step*.pt"),
                       key=lambda p: int(p.stem.split("_step")[1].split("_tokens")[0]))
    if existing:
        latest = existing[-1]
        print(f"Resuming from {latest.name}")
        ckpt = load_checkpoint(latest, map_location=DEVICE)
        assert ckpt["config"] == MODEL_A_CONFIG, "Checkpoint config does not match MODEL_A_CONFIG in this notebook!"
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optim_state"])
        scaler.load_state_dict(ckpt["scaler_state"])
        torch.set_rng_state(ckpt["rng_state"].cpu())
        start_step = ckpt["step"]
        start_tokens_seen = ckpt["tokens_seen"]
        print(f"Resumed at step={start_step}, tokens_seen={start_tokens_seen:,}")
    else:
        print("legallm-checkpoints is attached but has no A_step*.pt files yet -- starting fresh.")

    # Carry forward the attached dataset's checkpoint(s) into this
    # session's own CHECKPOINT_DIR now, once, so the very first periodic
    # push in Section 10 already includes them rather than only this
    # session's own new checkpoints (prune_old_checkpoints will trim
    # these away too, once this session has saved enough of its own).
    for p in existing:
        dest = CHECKPOINT_DIR / p.name
        if not dest.exists():
            shutil.copy(p, dest)
else:
    print("No legallm-checkpoints dataset attached yet -- starting fresh (expected for the very first run).")


legallm-checkpoints is attached but has no A_step*.pt files yet -- starting fresh.


/tmp/ipykernel_23/3403536661.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


## 10. Training loop

Stops on whichever comes first: the time budget, `MAX_STEPS`, or (smoke
mode only) `MAX_STEPS` being a genuinely small number rather than the
budget. Always saves a final checkpoint on exit, even if the loop stops
on the time budget mid-run -- losing the last several minutes of
training to an unsaved run would be a waste of GPU-hours.

**Live progress:** a `tqdm` bar tracks elapsed time against
`TIME_BUDGET_MIN` (the real stop condition here, not a step count --
this data pipeline streams continuously rather than in fixed epochs, so
there's no natural "epoch" to bar against). Its postfix carries the
current wall-clock timestamp, step, loss, learning rate, and tokens
seen, refreshed every step. This same pattern -- timestamped progress
bar in every training loop -- is the standard for every notebook after
this one (B, C, D, S, and Phases 2-7), not just Model A.

**Checkpoints push to Kaggle as training runs, not only at the end**
(Section 8's fix): every `CHECKPOINT_EVERY_STEPS`, this loop prunes old
local checkpoints and pushes the current one. A session that dies
mid-run now loses at most one checkpoint interval of progress, not the
entire run -- this is exactly the failure mode that lost an earlier
session's several hours of training.


In [10]:
from tqdm.auto import tqdm
import datetime

run_id = f"{ARCH_ID}_{'smoke' if SMOKE_MODE else 'main'}_{time.strftime('%Y%m%d_%H%M%S')}"
print(f"run_id = {run_id}")

model.train()
data_iter = iter(train_loader)
step = start_step
tokens_seen = start_tokens_seen
t_start = time.time()
loss_history = []

# The bar's progress axis is minutes of the time budget spent, not steps
# -- MAX_STEPS is a large placeholder in main mode (the real run spans
# multiple resumed sessions, so there's no session-local step target),
# so a step-based bar would sit at 0% for the entire run. Time budget is
# the number that actually reflects "how close is this session to done".
progress_bar = tqdm(total=round(TIME_BUDGET_MIN, 1), unit="min",
                     desc=run_id, dynamic_ncols=True)
last_reported_min = 0.0

while step < MAX_STEPS:
    elapsed_min = (time.time() - t_start) / 60
    if elapsed_min >= TIME_BUDGET_MIN:
        progress_bar.close()
        print(f"\nTime budget ({TIME_BUDGET_MIN} min) reached at step {step}.")
        break

    lr = get_lr(step)
    for group in optimizer.param_groups:
        group["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0
    for _ in range(GRAD_ACCUM_STEPS):
        x, y = next(data_iter)
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1)) / GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()
        accum_loss += loss.item()
        tokens_seen += x.numel()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
    scaler.step(optimizer)
    scaler.update()

    loss_history.append(accum_loss)
    assert math.isfinite(accum_loss), "Non-finite loss during real training -- stop and investigate before continuing."
    if step % 20 == 0 or step == start_step:
        print(f"  step {step:6d}  loss={accum_loss:.4f}  lr={lr:.2e}  "
              f"tokens_seen={tokens_seen:,}  elapsed={elapsed_min:.1f} min")

    # Advance the bar by however many minutes actually passed since the
    # last update, and refresh its live status line every step.
    now_elapsed_min = (time.time() - t_start) / 60
    progress_bar.update(now_elapsed_min - last_reported_min)
    last_reported_min = now_elapsed_min
    progress_bar.set_postfix({
        "time": datetime.datetime.now().strftime("%H:%M:%S"),
        "step": step,
        "loss": f"{accum_loss:.3f}",
        "lr": f"{lr:.1e}",
        "tokens": f"{tokens_seen:,}",
    })

    step += 1
    if step % CHECKPOINT_EVERY_STEPS == 0:
        path = checkpoint_path_for(step, tokens_seen)
        save_checkpoint(path, model, optimizer, None, scaler, step, tokens_seen, MODEL_A_CONFIG)
        print(f"  [checkpoint saved -> {path.name}]")
        prune_old_checkpoints(keep=2)
        push_checkpoint_dataset(f"{run_id}: step={step}, tokens_seen={tokens_seen:,} (in progress)")
else:
    # The while condition (step < MAX_STEPS) failed rather than the
    # time-budget break firing -- close the bar here too so it doesn't
    # linger open with a stale postfix.
    progress_bar.close()

# Always save a final checkpoint, whatever the stop reason was.
final_path = checkpoint_path_for(step, tokens_seen)
save_checkpoint(final_path, model, optimizer, None, scaler, step, tokens_seen, MODEL_A_CONFIG)
total_min = (time.time() - t_start) / 60
print(f"\nStopped at step={step}, tokens_seen={tokens_seen:,}, after {total_min:.1f} min "
      f"(finished {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}).")
print(f"Final checkpoint -> {final_path.name}")


run_id = A_main_20260928_173037


A_main_20260928_173037:   0%|          | 0/300 [00:00<?, ?min/s]

  step      0  loss=10.5776  lr=1.50e-06  tokens_seen=65,536  elapsed=0.0 min
  step     20  loss=8.3511  lr=3.15e-05  tokens_seen=1,376,256  elapsed=7.3 min
  step     40  loss=7.0395  lr=6.15e-05  tokens_seen=2,686,976  elapsed=14.8 min
  step     60  loss=5.7491  lr=9.15e-05  tokens_seen=3,997,696  elapsed=22.2 min
  step     80  loss=5.2894  lr=1.21e-04  tokens_seen=5,308,416  elapsed=29.7 min
  step    100  loss=5.0524  lr=1.51e-04  tokens_seen=6,619,136  elapsed=37.1 min
  step    120  loss=4.7303  lr=1.81e-04  tokens_seen=7,929,856  elapsed=44.6 min
  step    140  loss=4.2361  lr=2.11e-04  tokens_seen=9,240,576  elapsed=52.1 min
  step    160  loss=4.4540  lr=2.41e-04  tokens_seen=10,551,296  elapsed=59.6 min
  step    180  loss=4.1577  lr=2.71e-04  tokens_seen=11,862,016  elapsed=67.1 min
  step    200  loss=4.0026  lr=3.00e-04  tokens_seen=13,172,736  elapsed=74.6 min
  step    220  loss=3.8907  lr=3.00e-04  tokens_seen=14,483,456  elapsed=82.0 min
  step    240  loss=3.7800  

**Manual download link, independent of the Kaggle Dataset push.**
The push to `legallm-checkpoints` (Section 8/15) is the normal way to
carry a checkpoint into the next session, but it depends on the
`KAGGLE_API_TOKEN` secret being toggled on -- which has already failed
silently once this project. This cell gives a direct, one-click download
link to the checkpoint file itself, straight to your computer, as a
safety net that doesn't depend on that secret working at all.


In [11]:
from IPython.display import FileLink, display

print("Download the final checkpoint directly (click the link below):")
display(FileLink(str(final_path.relative_to(Path.cwd())) if final_path.is_relative_to(Path.cwd()) else str(final_path)))
print(f"({final_path.stat().st_size / 1e6:.1f} MB -- this is model + optimizer state, "
      f"so it's the full checkpoint, not just the model weights.)")


Download the final checkpoint directly (click the link below):


/kaggle/working/phase1_model_a/checkpoints/A_step798_tokens52297728.pt

(1952.1 MB -- this is model + optimizer state, so it's the full checkpoint, not just the model weights.)


## 11. Validation loss on the dev set

A cheap, fast correctness/progress signal -- teacher-forced cross-entropy
on `dev_prompts.json` + `dev_truth.json` (the continuation each prompt is
paired with). This is **not** the BERTScore/ROUGE/BLEU metrics
`evaluate_phase1.py` computes -- those need actual generation, which
Section 11 handles -- but a falling val loss is the fastest sanity check
that training is working before spending time on generation.


In [12]:
with open(INPUT_DATA_DIR / "dev_prompts.json") as f:
    dev_prompts = json.load(f)
with open(INPUT_DATA_DIR / "dev_truth.json") as f:
    dev_truth = json.load(f)

# evaluate_phase1.py (the capstone's own local evaluator) requires both
# files to be a JSON OBJECT mapping ID strings -> {"text": "..."} -- not a
# list of records. p1_00_prep.ipynb was built to write them in exactly
# that shape, so we assert it here rather than silently mis-reading a
# different format.
assert isinstance(dev_prompts, dict) and isinstance(dev_truth, dict), (
    "Expected dev_prompts.json/dev_truth.json to be {id: {\"text\": ...}} objects, "
    f"got {type(dev_prompts).__name__}/{type(dev_truth).__name__}. Check p1_00_prep.ipynb's output format."
)
dev_ids = sorted(set(dev_prompts) & set(dev_truth))
print(f"Loaded {len(dev_prompts)} dev prompts, {len(dev_truth)} truth records, {len(dev_ids)} shared IDs.")
sample_id = dev_ids[0]
print(f"\nSample prompt [{sample_id}]:", json.dumps(dev_prompts[sample_id], indent=2)[:300])
print(f"Sample truth  [{sample_id}]:", json.dumps(dev_truth[sample_id], indent=2)[:300])

model.eval()
val_losses = []
with torch.no_grad():
    for doc_id in dev_ids[:200]:  # a fixed subsample -- fast, and enough to trend
        prompt_text = dev_prompts[doc_id]["text"]
        truth_text = dev_truth[doc_id]["text"]
        full_text = prompt_text + " " + truth_text
        ids = tokenizer.encode(full_text).ids[:TRAIN_SEQ_LEN]
        if len(ids) < 2:
            continue
        ids_t = torch.tensor([ids], device=DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(ids_t)
            loss = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)), ids_t[:, 1:].reshape(-1))
        val_losses.append(loss.item())

val_loss = sum(val_losses) / len(val_losses) if val_losses else float("nan")
print(f"\nDev-set teacher-forced loss over {len(val_losses)} examples: {val_loss:.4f}")
model.train()


Loaded 1000 dev prompts, 1000 truth records, 1000 shared IDs.

Sample prompt [acts_00000]: {
  "text": "Maharashtra Land Revenue Code, 1966 Maharashtra Act No. 41 of 1966 Last Updated 13th November, 2019 For Statement of Objects and Reasons, see Maharashtra Government Gazette, 1965, Part V Extraordinary pages 756-763, for Report of the Joint Committee, see ibid, 1966, Part V, pages 219-35
Sample truth  [acts_00000]: {
  "text": "interests vested in a person or aggregate of persons capable of holding the same; (9) \"farm building\" means a structure erected on land assessed or held for the purpose of agriculture for all or any of the following purposes connected with such land or any other land belonging to or c

Dev-set teacher-forced loss over 200 examples: 3.4655


MiniGPT(
  (embed): Embedding(32000, 896)
  (blocks): ModuleList(
    (0-15): 16 x Block(
      (attn_norm): RMSNorm()
      (attn): GQAAttention(
        (q_proj): Linear(in_features=896, out_features=896, bias=False)
        (k_proj): Linear(in_features=896, out_features=128, bias=False)
        (v_proj): Linear(in_features=896, out_features=128, bias=False)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
      )
      (mlp_norm): RMSNorm()
      (mlp): SwiGLU(
        (gate_proj): Linear(in_features=896, out_features=2432, bias=False)
        (up_proj): Linear(in_features=896, out_features=2432, bias=False)
        (down_proj): Linear(in_features=2432, out_features=896, bias=False)
      )
    )
  )
  (norm_f): RMSNorm()
  (head): Linear(in_features=896, out_features=32000, bias=False)
)

## 12. Generate predictions & pack a submission

Predictions are written in the exact shape `evaluate_phase1.py` requires:
a JSON object mapping the same IDs `dev_truth.json` uses to
`{"text": "<generated continuation>"}`. Since this only generates for a
sample of IDs (not the full dev set -- see `GENERATION_SAMPLE_SIZE`
below), Section 13 needs a truth file trimmed to that same sample --
`evaluate_phase1.py` hard-fails on any ID mismatch between truth and
prediction, so we build that subset here rather than leaving it for
Section 13 to discover as an error.

**Generation is one-token-at-a-time and genuinely slow at this model
size** -- generating for the *entire* dev set (up to ~1,000 examples x
128 tokens each) can easily cost 1-3 hours on its own, on top of
training. `GENERATION_SAMPLE_SIZE` is capped well below the full dev set
even outside smoke mode for exactly this reason; only widen it once
you're generating a real submission from a checkpoint worth scoring, not
on every training session.


In [13]:
GENERATION_SAMPLE_SIZE = 20 if SMOKE_MODE else min(100, len(dev_ids))  # capped even in main mode -- see note above
MAX_NEW_TOKENS = 128
sample_ids = dev_ids[:GENERATION_SAMPLE_SIZE]

model.eval()
predictions = {}
t0 = time.time()
for doc_id in sample_ids:
    prompt_ids = tokenizer.encode(dev_prompts[doc_id]["text"]).ids
    prompt_tensor = torch.tensor([prompt_ids], device=DEVICE)
    with torch.no_grad():
        out_ids = model.generate(prompt_tensor, max_new_tokens=MAX_NEW_TOKENS, kv_cache=True, sampler="greedy")
    generated_text = tokenizer.decode(out_ids[0, len(prompt_ids):].tolist())
    predictions[doc_id] = {"text": generated_text if generated_text.strip() else "."}
    # evaluate_phase1.py's validate() rejects an empty/whitespace-only
    # "text" field outright -- a very undertrained smoke-run model can
    # genuinely generate nothing, so this keeps that edge case from
    # crashing Section 12 instead of silently producing a wrong score.
model.train()

elapsed = time.time() - t0
print(f"Generated {len(predictions)} predictions in {elapsed:.1f}s ({elapsed / max(1, len(predictions)):.2f}s/example).")
print("\nFirst prediction:", json.dumps({sample_ids[0]: predictions[sample_ids[0]]}, indent=2)[:300])

predictions_path = OUTPUT_DIR / "dev_predictions.json"
with open(predictions_path, "w") as f:
    json.dump(predictions, f, indent=2)

# The truth subset matching exactly the IDs we generated for -- required
# by evaluate_phase1.py's set(truth) == set(prediction) check.
truth_subset_path = OUTPUT_DIR / "dev_truth_subset.json"
with open(truth_subset_path, "w") as f:
    json.dump({doc_id: dev_truth[doc_id] for doc_id in sample_ids}, f, indent=2)

print(f"\nSaved -> {predictions_path}")
print(f"Saved -> {truth_subset_path} (matching truth subset, for Section 12)")


Generated 100 predictions in 164.7s (1.65s/example).

First prediction: {
  "acts_00000": {
    "text": " land; (2) the land in which the land is situated; (2) the land in the area of the land; (2) the land in the land; (3) the land in the land; (3) the land in the area of the land; (3) the land in the land; (3) the land in the land; (3) the land in the land; (3) the la

Saved -> /kaggle/working/phase1_model_a/dev_predictions.json
Saved -> /kaggle/working/phase1_model_a/dev_truth_subset.json (matching truth subset, for Section 12)


## 13. Score locally with evaluate_phase1.py's own metrics

This is the step that replaces submitting the smoke run to Codabench
(see the plan doc). Rather than shelling out to `evaluate_phase1.py` as a
separate file -- which would mean attaching it as yet another Kaggle
input just for this -- the same algorithms are reproduced directly below
(BERTScore, ROUGE-L, BLEU-4, and the exact ID-matching/validation rules),
so this cell gives the same numbers Codabench's own scoring would, without
spending one of the 6 allowed submissions on a deliberately undertrained
model.


In [14]:
import math
from collections import Counter

try:
    import bert_score
except ImportError:
    print("Installing bert-score (one-time, this session)...")
    subprocess.run(["pip", "install", "-q", "bert-score"], check=True)  # subprocess already imported in Section 8
    import bert_score


def validate_eval_format(data, name):
    """Mirrors evaluate_phase1.py's validate(): every record must be a
    non-empty string ID mapping to a dict with a non-empty string "text".
    """
    if not isinstance(data, dict) or not data:
        raise ValueError(f"{name} must be a non-empty JSON object mapping IDs to records.")
    for doc_id, rec in data.items():
        if (not isinstance(doc_id, str) or not doc_id.strip() or not isinstance(rec, dict)
                or "text" not in rec or not isinstance(rec["text"], str) or not rec["text"].strip()):
            raise ValueError(f"Invalid record for ID {doc_id!r} in {name}; each record needs a non-empty string field 'text'.")


def lcs_length(a, b):
    """Bit-parallel longest-common-subsequence length -- identical to
    evaluate_phase1.py's implementation, so ROUGE-L matches exactly.
    """
    if not a or not b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    masks = {}
    for j, t in enumerate(a):
        masks[t] = masks.get(t, 0) | (1 << j)
    state = 0
    for t in b:
        m = masks.get(t, 0)
        x = m | state
        y = (state << 1) | 1
        state = x & ~(x - y)
    return state.bit_count() if hasattr(int, "bit_count") else bin(state).count("1")


def rouge_l_f1(pred_text, ref_text):
    pt, rt = pred_text.split(), ref_text.split()
    if not pt or not rt:
        return 0.0
    l = lcs_length(pt, rt)
    if not l:
        return 0.0
    precision, recall = l / len(pt), l / len(rt)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)


def ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)) if len(tokens) >= n else Counter()


def bleu4(preds, refs):
    order = 4
    clipped, totals = [0] * order, [0] * order
    pred_len, ref_len = 0, 0
    for pred_tokens, ref_tokens in zip(preds, refs):
        pred_len += len(pred_tokens)
        ref_len += len(ref_tokens)
        for n in range(1, order + 1):
            pred_ngrams, ref_ngrams = ngrams(pred_tokens, n), ngrams(ref_tokens, n)
            totals[n - 1] += sum(pred_ngrams.values())
            for gram, count in pred_ngrams.items():
                clipped[n - 1] += min(count, ref_ngrams.get(gram, 0))
    precisions = [clipped[i] / totals[i] if totals[i] else 0.0 for i in range(order)]
    if any(p <= 0 for p in precisions) or pred_len == 0:
        return 0.0
    brevity_penalty = 1.0 if pred_len > ref_len else math.exp(1.0 - ref_len / pred_len)
    return brevity_penalty * math.exp(sum(math.log(p) for p in precisions) / order)


def score_predictions(pred_dict, truth_dict):
    """Same contract as evaluate_phase1.py's main(): validates both
    inputs, requires an exact ID match, then returns (bertscore, rouge_l,
    bleu4) over the shared IDs.
    """
    validate_eval_format(truth_dict, "truth")
    validate_eval_format(pred_dict, "prediction")
    if set(truth_dict) != set(pred_dict):
        missing = len(set(truth_dict) - set(pred_dict))
        extra = len(set(pred_dict) - set(truth_dict))
        raise ValueError(f"ID mismatch: missing={missing}, extra={extra}")

    ordered_ids = list(truth_dict)
    pred_texts = [pred_dict[i]["text"].strip() for i in ordered_ids]
    ref_texts = [truth_dict[i]["text"].strip() for i in ordered_ids]
    pred_tokens = [t.split() for t in pred_texts]
    ref_tokens = [t.split() for t in ref_texts]

    eval_device = "cuda" if torch.cuda.is_available() else "cpu"
    _, _, f1 = bert_score.score(pred_texts, ref_texts, lang="en", batch_size=8, verbose=False, device=eval_device)
    bertscore = float(f1.mean().item())
    rouge = sum(rouge_l_f1(p, r) for p, r in zip(pred_texts, ref_texts)) / len(ordered_ids)
    b4 = bleu4(pred_tokens, ref_tokens)
    return bertscore, rouge, b4


with open(predictions_path) as f:
    _pred_for_scoring = json.load(f)
with open(truth_subset_path) as f:
    _truth_for_scoring = json.load(f)

try:
    dev_bertscore, dev_rouge_l, dev_bleu4 = score_predictions(_pred_for_scoring, _truth_for_scoring)
    print(f"Examples:     {len(_truth_for_scoring)}")
    print(f"BERTScore F1: {dev_bertscore:.6f}")
    print(f"ROUGE-L F1:   {dev_rouge_l:.6f}")
    print(f"BLEU-4:       {dev_bleu4:.6f}")
    if SMOKE_MODE:
        print("\n(Smoke run on very few generated tokens/examples -- low scores here are "
              "expected and don't mean anything about the real run. This cell passing "
              "without an exception is the actual pipeline-check signal.)")
except Exception as exc:
    dev_bertscore, dev_rouge_l, dev_bleu4 = None, None, None
    print(f"Scoring failed: {exc}")
    print("This is exactly the kind of format bug we wanted to catch before a real Codabench submission.")


Installing bert-score (one-time, this session)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.3 MB/s eta 0:00:00


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Examples:     100
BERTScore F1: 0.823718
ROUGE-L F1:   0.154756
BLEU-4:       0.028251


## 14. Append this run to `runs.csv`

Columns match the plan doc's contract exactly:
`run_id, arch_id, technique, params, tokens_seen, tokens_per_sec,
peak_mem_gb, val_loss, bits_per_byte, dev_bertscore, dev_rouge_l,
dev_bleu4, codabench_score, checkpoint_path, notes`. BERTScore/ROUGE-L/
BLEU-4 now come from Section 12's local scoring; `codabench_score` stays
blank here since this run was never actually submitted.


In [15]:
import csv

RUNS_CSV_PATH = OUTPUT_DIR / "runs.csv"
RUNS_CSV_COLUMNS = [
    "run_id", "arch_id", "technique", "params", "tokens_seen", "tokens_per_sec",
    "peak_mem_gb", "val_loss", "bits_per_byte", "dev_bertscore", "dev_rouge_l",
    "dev_bleu4", "codabench_score", "checkpoint_path", "notes",
]

tokens_per_sec = tokens_seen / (total_min * 60) if total_min > 0 else None
peak_mem_gb = torch.cuda.max_memory_allocated() / 1e9

row = {
    "run_id": run_id,
    "arch_id": ARCH_ID,
    "technique": "full-pretrain",
    "params": n_params,
    "tokens_seen": tokens_seen,
    "tokens_per_sec": round(tokens_per_sec, 1) if tokens_per_sec else None,
    "peak_mem_gb": round(peak_mem_gb, 2),
    "val_loss": round(val_loss, 4) if val_losses else None,
    "bits_per_byte": None,
    "dev_bertscore": round(dev_bertscore, 4) if dev_bertscore is not None else None,
    "dev_rouge_l": round(dev_rouge_l, 4) if dev_rouge_l is not None else None,
    "dev_bleu4": round(dev_bleu4, 4) if dev_bleu4 is not None else None,
    "codabench_score": None,
    "checkpoint_path": str(final_path.name),
    "notes": f"{'smoke pipeline-check run' if SMOKE_MODE else 'main run'}; grad checkpointing on; seq_len={TRAIN_SEQ_LEN}",
}

file_exists = RUNS_CSV_PATH.exists()
with open(RUNS_CSV_PATH, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=RUNS_CSV_COLUMNS)
    if not file_exists:
        writer.writeheader()
    writer.writerow(row)

print(f"Appended run to {RUNS_CSV_PATH}:")
print(json.dumps(row, indent=2))


Appended run to /kaggle/working/phase1_model_a/runs.csv:
{
  "run_id": "A_main_20260928_173037",
  "arch_id": "A",
  "technique": "full-pretrain",
  "params": 162657152,
  "tokens_seen": 52297728,
  "tokens_per_sec": 2901.5,
  "peak_mem_gb": 11.57,
  "val_loss": 3.4655,
  "bits_per_byte": null,
  "dev_bertscore": 0.8237,
  "dev_rouge_l": 0.1548,
  "dev_bleu4": 0.0283,
  "codabench_score": null,
  "checkpoint_path": "A_step798_tokens52297728.pt",
  "notes": "main run; grad checkpointing on; seq_len=1024"
}


## 15. Final publish: checkpoint + predictions + runs.csv as a Kaggle Dataset

Reuses Section 8's `push_checkpoint_dataset()` -- the same function
Section 10 already called periodically during training -- so there's
only one push implementation to trust, not two that could drift apart.
The one thing specific to this final call: merging `runs.csv` history
from the attached dataset's previous version, so the run log
accumulates across sessions instead of resetting each time (checkpoints
and predictions don't need merging -- they're just files sitting in
OUTPUT_DIR already).


In [16]:
# If a previous version of this dataset had a runs.csv, merge this run
# into it rather than starting a fresh history each version.
if INPUT_CHECKPOINT_DIR.exists():
    prev_runs_csv = INPUT_CHECKPOINT_DIR / "runs.csv"
    if prev_runs_csv.exists():
        prev_rows = list(csv.DictReader(open(prev_runs_csv)))
        with open(RUNS_CSV_PATH) as f:
            new_rows = list(csv.DictReader(f))
        merged = prev_rows + new_rows
        with open(RUNS_CSV_PATH, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=RUNS_CSV_COLUMNS)
            writer.writeheader()
            writer.writerows(merged)
        print(f"Merged {len(prev_rows)} previous run(s) from the attached dataset's runs.csv.")

pushed = push_checkpoint_dataset(f"{run_id} final: step={step}, tokens_seen={tokens_seen:,}, "
                                  f"val_loss={val_loss:.4f}")
if pushed:
    print(f"Done. Attach '{CHECKPOINT_DATASET_SLUG}' as input to resume this run in a later session.")
else:
    print(f"Checkpoint, predictions, and runs.csv are still saved locally under {OUTPUT_DIR} "
          f"-- download them manually if needed before the session ends.")


Merged 1 previous run(s) from the attached dataset's runs.csv.
Pushed checkpoint dataset (A_main_20260928_173037 final: step=798, tokens_seen=52,297,728, val_loss=3.4655).
Done. Attach 'legallm-checkpoints' as input to resume this run in a later session.


## 16. Summary

| Item | Where |
| --- | --- |
| Model A parameter count | printed in Section 4 -- must be 50-300M |
| Fitting micro-batch size at `TRAIN_SEQ_LEN` w/ checkpointing | printed in Section 5 |
| Checkpoints pushed *during* training, not just at the end | Section 8 (helper), called every `CHECKPOINT_EVERY_STEPS` from Section 10 |
| Final step / tokens seen this run | printed in Section 10 |
| Dev-set val loss | printed in Section 11 |
| `dev_predictions.json` | Section 12 |
| Local BERTScore/ROUGE-L/BLEU-4 | Section 13 -- **this replaces a Codabench submission for the smoke run** |
| `runs.csv` row | Section 14 |
| Final checkpoint publish | Section 15, dataset `legallm-checkpoints` |

**If this was a smoke run (`SMOKE_MODE=True`):** once it completes end to
end -- checkpoint saves, `legallm-checkpoints` publishes, and Section 13
scores `dev_predictions.json` without erroring -- the pipeline is proven.
No Codabench submission needed for this run; save all 6 for real
checkpoints. Then flip `SMOKE_MODE=False` and re-attach
`legallm-checkpoints` as input for tomorrow's real run, which
resumes from exactly where this one stopped -- and now survives a
session dying mid-run too, since Section 10 pushes checkpoints as it
goes rather than only once at the very end.

**Follow-up noted, not done here:** the plan doc's checkpoint contract
calls for the `legallm` package itself to eventually be published as its
own `legallm-src` Kaggle Dataset so code isn't copy-pasted between
notebooks. Worth doing once the model/block interfaces stop changing
notebook-to-notebook -- premature right after p1_02 and p1_03 were still
finding bugs in exactly this code.
